# WarpKriging with affine warping (Octave)

The **affine** warp applies a learned linear transformation $w(x) = a \cdot x + b$ to each input.
This can help when the inputs need rescaling or shifting beyond simple normalization.

Steps:
1. Setup mlibkriging / jlibkriging
2. Define the Branin function and plot it
3. Build a space-filling design and evaluate it
4. Fit a `WarpKriging` model
5. Predict on a fine grid and plot mean + uncertainty
6. Inspect model parameters

## 0. Setup

Build the C++ core and the Octave binding from source (skip if already built).
Requires: `cmake`, a C++ compiler, Octave ≥ 6.0.

In [1]:
% Add mlibkriging to path
% Adjust the path below to your build/installed directory
repo_root = fullfile(fileparts(pwd()), '..');
build_path = fullfile(repo_root, 'build', 'installed', 'bindings', 'Octave');
if ~exist(fullfile(build_path, 'mLibKriging.mex'), 'file') ...
   && ~exist(fullfile(build_path, ['mLibKriging.', mexext]), 'file')
    error('mlibkriging not found at %s — please build first (see README.md)', build_path);
end
addpath(build_path);
addpath(fullfile(repo_root, 'bindings', 'Octave', 'mlibkriging'));
disp('mlibkriging loaded')

mlibkriging loaded


## 1. Branin function

The Branin function is a standard benchmark for surrogate modelling, defined on $[0,1]^2$
(rescaled from its canonical domain $[-5, 10] \times [0, 15]$).
It has three global minima.

In [1]:
function z = branin(X)
    x1 = X(:,1) * 15 - 5;
    x2 = X(:,2) * 15;
    z = (x2 - 5/(4*pi^2) * x1.^2 + 5/pi * x1 - 6).^2 ...
        + 10 * (1 - 1/(8*pi)) * cos(x1) + 10;
end

% 50x50 evaluation grid
grid_x = linspace(0, 1, 50);
[G1, G2] = meshgrid(grid_x, grid_x);
grid_pts = [G1(:), G2(:)];
z_true = reshape(branin(grid_pts), 50, 50);

figure;
contourf(G1, G2, z_true, 20);
colorbar;
title('True Branin function');
xlabel('x_1'); ylabel('x_2');

## 2. Design of experiments

We sample $n = 30$ points using a Latin Hypercube Design.

In [2]:
rand('seed', 42);
n = 30; d = 2;

% Simple LHS: stratified uniform sample, independently permuted per dimension
X = zeros(n, d);
for j = 1:d
    perm = randperm(n);
    X(:,j) = (perm' - rand(n,1)) / n;
end
y = branin(X);

figure;
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title(sprintf('%d LHS design points on Branin', n));
xlabel('x_1'); ylabel('x_2');

## 3. Fit a WarpKriging model (`affine`)

We use `warping = {'affine', 'affine'}` — one warp per input dimension.

In [3]:
wk = WarpKriging(y, X, {'affine', 'affine'}, 'matern5_2', 'constant', false, 'BFGS+Adam', 'LL');
disp(wk.summary())

* WarpKriging
* data: 30x[0.00734027,0.989352],[0.0212626,0.988936] -> 30x[1.35568,276.199]
* trend constant (est.): 1812.76
* variance (est.): 3.59989e+06
* covariance:
  * kernel: matern5_2
  * range (est.): 1.70976, 5.39401
  * warpings:
      x0: "affine"  →  Affine(a=1.001, b=-4.85457e-05)
      x1: "affine"  →  Affine(a=0.999, b=-6.38464e-05)
  * total warp params: 4
  * fit:
    * objective: LL
    * optim: BFGS+Adam



## 4. Predict on a fine grid

`predict()` returns the posterior mean and standard deviation at new points.

In [4]:
[p_mean, p_stdev] = wk.predict(grid_pts, true, false);
z_mean = reshape(p_mean, 50, 50);
z_sd   = reshape(p_stdev, 50, 50);

vmin = min(min(z_true(:)), min(z_mean(:)));
vmax = max(max(z_true(:)), max(z_mean(:)));

figure;
subplot(1, 2, 1);
contourf(G1, G2, z_true, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('True Branin'); xlabel('x_1'); ylabel('x_2');

subplot(1, 2, 2);
contourf(G1, G2, z_mean, 20);
hold on;
scatter(X(:,1), X(:,2), 40, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
caxis([vmin vmax]); colorbar;
title('WarpKriging (affine) mean'); xlabel('x_1'); ylabel('x_2');

In [5]:
% Posterior standard deviation (uncertainty)
figure;
contourf(G1, G2, z_sd, 20);
hold on;
scatter(X(:,1), X(:,2), 60, 'w', 'filled', 'MarkerEdgeColor', 'k');
hold off;
colorbar;
title('WarpKriging (affine) std dev (uncertainty)');
xlabel('x_1'); ylabel('x_2');

## 5. Model inspection

Key fitted parameters: length-scales $\theta$, variance $\sigma^2$, log-likelihood, and warping specification.

In [6]:
fprintf('Kernel       : %s\n', wk.kernel());
fprintf('Theta (range): %s\n', mat2str(wk.theta(), 4));
fprintf('Sigma2       : %.4f\n', wk.sigma2());
fprintf('LogLikelihood: %.4f\n', wk.logLikelihood());
fprintf('Feature dim  : %d\n', wk.feature_dim());
fprintf('Warping      : ');
w = wk.warping(); for i = 1:length(w); fprintf('%s ', w{i}); end; fprintf('\n');

Kernel       : matern5_2


Theta (range): [1.71;5.394]


Sigma2       : 3599892.9832


LogLikelihood: -99.3352


Feature dim  : 2


Warping      : 

affine affine 
